# 02 | ASCII Graphs and QCTN

This notebook answers one central question: **how does an ASCII diagram become an executable tensor network?**

You will learn to write graphs, inspect inferred shapes, generate MPS and brickwall topologies, and reason about boundaries, bonds, and empty wires.


In [1]:
from pathlib import Path
import sys

# This works whether Jupyter starts in the repository root or in notebooks/.
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "tneq_qc").is_dir() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/yuch3n/Documents/Code/Github/tneq-qc


In [2]:
from tneq_qc import BackendFactory, QCTN, QCTNHelper

backend = BackendFactory.create_backend("pytorch", device="cpu", dtype="float32")


## 1. Basic graph syntax

Consider one row:

```text
-2-A-4-B-2-
```

- `A` and `B` are core symbols.
- The left `2` is an input boundary dimension for `A`.
- The middle `4` is a shared bond between `A` and `B` on this qubit.
- The right `2` is an output boundary dimension for `B`.
- Dashes are separators and visual spacing.

Repeating a symbol on multiple rows means that one high-order core spans those qubits. Uppercase and lowercase symbols are distinct.


In [3]:
graph = """\
-2-A-4-B-2-
-2-A-4-B-2-"""

qctn = QCTN(graph, backend=backend).auto_init()
print(qctn)
print("\nRendered from the parsed structure:")
print(qctn.to_graph_string())


QCTN(nqubits=2, cores=[A(2, 2, 4, 4), B(4, 4, 2, 2)])

Rendered from the parsed structure:
-2-A-4-B-2-
-2-A-4-B-2-


In [4]:
# adjacency_table is the most useful structure for debugging a graph.
for info in qctn.adjacency_table:
    name = info["core_name"]
    print("Core:", name)
    print("  input_shape :", info["input_shape"])
    print("  output_shape:", info["output_shape"])
    print("  input_dim   :", info["input_dim"])
    print("  output_dim  :", info["output_dim"])
    print("  tensor shape:", qctn[name].shape)


Core: A
  input_shape : [2, 2]
  output_shape: [4, 4]
  input_dim   : 4
  output_dim  : 16
  tensor shape: torch.Size([2, 2, 4, 4])
Core: B
  input_shape : [4, 4]
  output_shape: [2, 2]
  input_dim   : 16
  output_dim  : 4
  tensor shape: torch.Size([4, 4, 2, 2])


## 2. Boundary dimensions versus bond dimensions

A boundary dimension usually represents the local physical space—for a qubit this is commonly 2. Internal bond dimensions control model capacity: larger bonds can encode richer correlations but increase parameter count, memory use, and contraction cost.

For each core, the initialized tensor shape is:

```python
tuple(input_shape + output_shape)
```

A core spanning several rows receives edges from each row and therefore has a higher tensor rank.


## 3. Built-in topology generators

Start with a generator whenever possible, then modify its output. The current `QCTNHelper.mps()` uses a staggered layout: an `n`-qubit MPS usually contains `n-1` bond cores rather than one core per qubit.


In [5]:
mps_graph = QCTNHelper.mps(nqubits=5, bond_dim=3, phys_dim=2)
print("MPS graph:\n", mps_graph)

mps = QCTN(mps_graph, backend=backend).auto_init(orthogonal=True)
print("\nParsed model:", mps)
for name in mps.cores:
    print(name, mps[name].shape)


MPS graph:
 -2-a-------------------2-
-2-a--3--b-------------2-
-2-------b--3--c-------2-
-2-------------c--3--d-2-
-2-------------------d-2-

Parsed model: QCTN(nqubits=5, cores=[a(2, 2, 2, 3), b(3, 2, 2, 3), c(3, 2, 2, 3), d(3, 2, 2, 2)])
a torch.Size([2, 2, 2, 3])
b torch.Size([3, 2, 2, 3])
c torch.Size([3, 2, 2, 3])
d torch.Size([3, 2, 2, 2])


In [6]:
wall_graph = QCTNHelper.brickwall(nqubits=4, n_layers=3, phys_dim=2)
print("Brickwall graph:\n", wall_graph)

wall = QCTN(wall_graph, backend=backend).auto_init(orthogonal=True)
print("\nNumber of brickwall cores:", wall.ncores)


Brickwall graph:
 -2-a-2-----d-2-
-2-a-2-c-2-d-2-
-2-b-2-c-2-e-2-
-2-b-2-----e-2-

Number of brickwall cores: 5


## 4. State and measurement graphs

A product-state ket has only an output edge:

```text
-A-2-
```

A local measurement matrix has both an input and output edge:

```text
-2-A-2-
```

These components will close boundaries and inject data in a BornMachine.


In [7]:
print("State graph:")
print(QCTNHelper.state(3, phys_dim=2))

print("\nMeasurement graph:")
print(QCTNHelper.measure_matrix(3, phys_dim=2))


State graph:
-a-2-
-b-2-
-c-2-

Measurement graph:
-2-a-2-
-2-b-2-
-2-c-2-


## 5. Empty qubit rows

If a row contains no core, QCTN injects a fixed identity core to keep that wire connected. Fixed identities cannot become trainable parameters and ordinary assignment will not overwrite them.


In [8]:
graph_with_empty_row = """\
-2-A-2-
--------
-2-B-2-"""

padded = QCTN(graph_with_empty_row, backend=backend).auto_init()
for symbol in padded.cores:
    core = padded[symbol]
    print(padded.core_names[symbol], "fixed=", core.is_fixed, "shape=", core.shape)


identity.q1 fixed= True shape= torch.Size([2, 2])
A fixed= False shape= torch.Size([2, 2])
B fixed= False shape= torch.Size([2, 2])


## 6. Common graph mistakes

- Giving the two ends of one bond different dimensions.
- Reusing a letter accidentally and creating a multi-qubit core.
- Assuming the number of qubits must equal the number of cores.
- Treating visual dash spacing as topology; core order and dimensions carry the meaning.
- Mixing `A` and `a` while intending them to be the same core.

### Exercises

1. Generate a six-qubit MPS with bond dimension 4 and print every core shape.
2. Generate a five-qubit, four-layer brickwall and identify which qubits each core spans.
3. Write a valid three-qubit graph with two cores and verify it with `to_graph_string()`.
